<a href="https://colab.research.google.com/github/marchedev2002/ia/blob/main/agente_inmobiliaria/agente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalamos versiones compatibles para evitar errores con AgentExecutor
!pip install -q "langchain<0.3.0" "langchain-community<0.3.0" "langchain-core<0.3.0" langchain-groq chromadb sentence-transformers pydantic

In [ ]:
import os
import getpass

# Intentamos leer desde los Secretos de Colab (ícono de la llave); si no existe, la pide en pantalla
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Ingresa tu GROQ_API_KEY (empieza con gsk_...): ")

print("Clave de Groq configurada correctamente.")

## 1. Módulo Relacional: Base de Datos Transaccional (SQLite)
Almacenamos los datos estructurados del sistema de administración inmobiliaria: contratos registrados, fechas de vencimiento, montos de canon y seguimiento de morosidad.

In [ ]:
import sqlite3

# Conexión local a la base SQLite en el entorno temporal de Colab
db_conn = sqlite3.connect("inmobiliaria_backoffice.db")
cursor = db_conn.cursor()

# 1. Tabla de contratos
cursor.execute("""
CREATE TABLE IF NOT EXISTS contratos (
    id_contrato TEXT PRIMARY KEY,
    direccion TEXT NOT NULL,
    inquilino_nombre TEXT NOT NULL,
    propietario_nombre TEXT NOT NULL,
    monto_canon REAL NOT NULL,
    fecha_inicio DATE NOT NULL,
    fecha_vencimiento DATE NOT NULL
);
""")

# 2. Tabla de cobros y mora
cursor.execute("""
CREATE TABLE IF NOT EXISTS pagos (
    id_pago INTEGER PRIMARY KEY AUTOINCREMENT,
    id_contrato TEXT NOT NULL,
    periodo TEXT NOT NULL,
    estado TEXT NOT NULL, -- 'AL_DIA', 'MORA'
    dias_atraso INTEGER DEFAULT 0,
    FOREIGN KEY(id_contrato) REFERENCES contratos(id_contrato)
);
""")

# Carga de datos de prueba representativos
contratos_mock = [
    ("LOC-2024-01", "Av. Corrientes 1240 4B", "Juan Perez", "Roberto Gomez", 350000.0, "2024-01-01", "2026-01-01"),
    ("LOC-2024-02", "Bv. Oroño 850 1A", "Lucia Fernandez", "Marta Lopez", 480000.0, "2024-03-01", "2026-03-01"),
    ("LOC-2024-03", "San Martin 520 PB", "Carlos Tevez", "Esteban Quito", 290000.0, "2023-11-01", "2025-11-01")
]

pagos_mock = [
    ("LOC-2024-01", "Septiembre 2024", "MORA", 12),
    ("LOC-2024-02", "Septiembre 2024", "AL_DIA", 0),
    ("LOC-2024-03", "Septiembre 2024", "MORA", 5)
]

cursor.executemany("INSERT OR REPLACE INTO contratos VALUES (?,?,?,?,?,?,?)", contratos_mock)
cursor.executemany("INSERT OR REPLACE INTO pagos (id_contrato, periodo, estado, dias_atraso) VALUES (?,?,?,?)", pagos_mock)
db_conn.commit()

print("Base de datos SQLite inicializada y cargada con éxito.")

## 2. Módulo RAG: Indexación Vectorial de Contratos Legales (ChromaDB)
Procesamos las cláusulas legales no estructuradas de los contratos. Se aplica chunking semántico delimitado por cláusulas y vectorización con un modelo de embeddings de HuggingFace que se ejecuta localmente sin costo.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Textos contractuales no estructurados
c1 = """
CONTRATO DE LOCACIÓN: LOC-2024-01 | Inmueble: Av. Corrientes 1240 4B
CLÁUSULA PRIMERA (DESTINO): Destino exclusivo vivienda familiar y permanente.
CLÁUSULA SEGUNDA (PAGO Y MORA): Vencimiento el día 10 de cada mes. La mora devengará un interés punitorio diario del 0.6% por cada día de retraso.
CLÁUSULA TERCERA (EXPENSAS Y REPARACIONES): Las expensas ordinarias corresponden al inquilino. Las extraordinarias y arreglos de cañerías maestras son a cargo del propietario.
CLÁUSULA CUARTA (RESCISIÓN ANTICIPADA): Requiere preaviso formal de 30 días e indemnización equivalente a 1 mes de canon de alquiler.
"""

c2 = """
CONTRATO DE LOCACIÓN: LOC-2024-02 | Inmueble: Bv. Oroño 850 1A
CLÁUSULA PRIMERA (DESTINO): Uso comercial o estudio profesional habilitado.
CLÁUSULA SEGUNDA (PAGO Y MORA): Vencimiento el día 5 de cada mes. Interés punitorio por mora pactado en 1.0% diario.
CLÁUSULA TERCERA (GARANTÍA): Fianza asumida solidariamente por Carlos Fernandez.
CLÁUSULA CUARTA (RESCISIÓN ANTICIPADA): Exige 60 días de preaviso y abono de 2 meses de canon indemnizatorio.
"""

c3 = """
CONTRATO DE LOCACIÓN: LOC-2024-03 | Inmueble: San Martin 520 PB
CLÁUSULA PRIMERA (DESTINO): Vivienda familiar sin admisión de sublocaciones.
CLÁUSULA SEGUNDA (PAGO Y MORA): Vencimiento el día 10. Tasa diaria punitoria por mora del 0.4% por día de retraso.
CLÁUSULA TERCERA (EXPENSAS): Expensas comunes por el locatario; fondos de reserva a cargo del locador.
"""

documentos = [
    Document(page_content=c1, metadata={"id_contrato": "LOC-2024-01", "direccion": "Av. Corrientes 1240 4B"}),
    Document(page_content=c2, metadata={"id_contrato": "LOC-2024-02", "direccion": "Bv. Oroño 850 1A"}),
    Document(page_content=c3, metadata={"id_contrato": "LOC-2024-03", "direccion": "San Martin 520 PB"})
]

# Segmentación semántica respetando delimitación de cláusulas
splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=50, separators=["CLÁUSULA ", "\n\n", "\n", " "])
chunks = splitter.split_documents(documentos)

# Embeddings gratuitos de HuggingFace en CPU
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings, collection_name="contratos_colab")
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"RAG inicializado: {len(chunks)} fragmentos indexados en ChromaDB.")

## 3. Catálogo de Herramientas (*Tools*) del Agente
Definimos las herramientas que permiten al agente interactuar con los distintos módulos:
1. `consultar_contrato_rag`: Búsqueda semántica documental de cláusulas legales.
2. `ejecutar_consulta_sql`: Motor de consulta relacional para datos de cobros y contratos.
3. `calcular_mora_exacta`: Cálculo aritmético preciso para liquidaciones financieras.
4. `redactar_intimacion`: Generación de notificaciones de cobranza.

In [ ]:
from langchain_core.tools import tool

@tool
def consultar_contrato_rag(consulta: str) -> str:
    """Busca cláusulas legales sobre mora pactada, rescisión, garantías o responsabilidades de expensas."""
    docs = retriever.invoke(consulta)
    if not docs:
        return "No se hallaron cláusulas relevantes para esa búsqueda."
    return "\n---\n".join([f"[{d.metadata.get('id_contrato')}]: {d.page_content.strip()}" for d in docs])

@tool
def ejecutar_consulta_sql(query_sql: str) -> str:
    """Ejecuta sentencias SQL de lectura (SELECT) en la base de la inmobiliaria.
    Esquema disponible:
    - contratos(id_contrato, direccion, inquilino_nombre, propietario_nombre, monto_canon, fecha_inicio, fecha_vencimiento)
    - pagos(id_pago, id_contrato, periodo, estado, dias_atraso)
    """
    if not query_sql.strip().upper().startswith("SELECT"):
        return "Error: Solo se autorizan consultas de tipo SELECT de solo lectura."
    try:
        cur = db_conn.cursor()
        cur.execute(query_sql)
        filas = cur.fetchall()
        columnas = [d[0] for d in cur.description]
        return f"Columnas: {columnas}\nFilas: {filas}"
    except Exception as e:
        return f"Error SQL: {str(e)}"

@tool
def calcular_mora_exacta(monto_canon: float, dias_mora: int, tasa_diaria_porcentaje: float) -> str:
    """Calcula matemáticamente el recargo por mora acumulada y el saldo total exigible."""
    interes = monto_canon * (tasa_diaria_porcentaje / 100.0) * dias_mora
    total = monto_canon + interes
    return (
        f"Monto Base: ${monto_canon:,.2f} | Días de atraso: {dias_mora} | Tasa: {tasa_diaria_porcentaje}%\n"
        f"Interés Punitorio: ${interes:,.2f}\n"
        f"TOTAL A LIQUIDAR: ${total:,.2f}"
    )

@tool
def redactar_intimacion(destinatario: str, direccion: str, detalle_deuda: str) -> str:
    """Genera la plantilla formal de reclamo de pago para el departamento de cobranzas."""
    return f"""
==================== NOTIFICACIÓN FORMAL DE INTIMACIÓN ====================
Destinatario: {destinatario}
Inmueble: {direccion}

Por medio de la presente intimamos a usted a cancelar en un plazo perentorio de 48 hs
la deuda devengada a la fecha, según el siguiente detalle:

{detalle_deuda}

Transcurrido el plazo sin regularización, se remitirán los antecedentes al departamento
jurídico para iniciar el cobro por vía ejecutiva y la citación a los garantes.

Atentamente,
Departamento de Cobranzas y Asuntos Legales.
===========================================================================
"""

tools = [consultar_contrato_rag, ejecutar_consulta_sql, calcular_mora_exacta, redactar_intimacion]
print("Herramientas operativas registradas:", [t.name for t in tools])

## 4. Orquestación del Agente ReAct (Tool-Calling con LLaMA 3.1)
El modelo analiza la consulta del analista y decide qué herramientas invocar de manera autónoma para resolver la tarea.

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# LLaMA 3.1 8B vía Groq API con soporte nativo de Tool Calling
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

system_prompt = """Eres 'InmoOps Copilot', un asistente analista de operaciones internas para el equipo de administración inmobiliaria.

Tus directivas operativas son:
1. Para datos de contratos, personas, fechas y deudas en mora: ejecuta consultas SQL con 'ejecutar_consulta_sql'.
2. Para cláusulas, reglas de convivencia o tasas pactadas: consulta el RAG con 'consultar_contrato_rag'.
3. Para cálculos financieros: NO hagas cálculos aritméticos directos; invoca siempre 'calcular_mora_exacta'.
4. Si se solicita armar una comunicación formal, usa 'redactar_intimacion'.
5. Sé claro, sintético y presenta la información de forma estructurada para el operador interno."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Agente InmoOps Copilot con LLaMA 3.1 ensamblado correctamente.")

## 5. Demostración Operativa y Casos de Prueba
Se ejecutan tres pruebas para validar el comportamiento del sistema experto:
1. Búsqueda contextual puramente documental (RAG).
2. Extracción analítica estructurada (SQL).
3. Pipeline agéntico integral (SQL -> RAG -> Math Tool -> Drafting Tool).

In [ ]:
res_1 = agent_executor.invoke({
    "input": "¿A cargo de quién corresponden las reparaciones de cañerías y las expensas extraordinarias en el contrato de Av. Corrientes 1240?"
})
print("\n--- RESPUESTA AL ANALISTA ---\n", res_1["output"])

In [ ]:
res_2 = agent_executor.invoke({
    "input": "¿Qué contratos registran pagos en estado de MORA en el sistema y cuántos días de retraso tiene cada uno?"
})
print("\n--- RESPUESTA AL ANALISTA ---\n", res_2["output"])

In [ ]:
res_3 = agent_executor.invoke({
    "input": (
        "Revisa si el contrato de Av. Corrientes 1240 tiene pagos pendientes este mes. "
        "Si está en mora, consulta en su contrato qué tasa de interés diario se pactó, "
        "calcula el total adeudado y déjame preparado el borrador de intimación formal."
    )
})
print("\n--- RESPUESTA AL ANALISTA ---\n", res_3["output"])

## 6. Documentación Técnica y Relación con TP1 (Criterios de Evaluación)

### Decisiones de Diseño
1. **Arquitectura Híbrida (RAG + SQL):** Se delimitó qué información es estructurada/transaccional (SQLite) y cuál interpretativa/documental (ChromaDB), reduciendo alucinaciones en fechas y saldos.
2. **Tools Determinísticas:** El LLM delega las operaciones aritméticas a código Python nativo para asegurar precisión contable en las liquidaciones de mora.
3. **Chunking Especializado:** Se aplicó segmentación respetando palabras clave (`CLÁUSULA`) para evitar fragmentar definiciones legales entre chunks contiguos.

### Vinculación con TP1 (Redes Neuronales)
- **TP1:** Se resolvieron problemas de clasificación mediante una red neuronal supervisada que optimizó pesos fijos mediante backpropagation sobre features tabulares.
- **TP2:** Se utiliza un modelo fundacional preentrenado (LLM) asistido por representaciones semánticas densas (embeddings) y orquestación dinámica de herramientas externas en tiempo de ejecución.